In [ ]:
import win32com.client
import pythoncom

In [ ]:
pythoncom.CoInitialize()
ol = win32com.client.Dispatch("Outlook.Application")

**Send Emails** <br>
import win32com.client <br>
ol=win32com.client.Dispatch("outlook.application") <br>
olmailitem=0x0 #size of the new email <br>
newmail=ol.CreateItem(olmailitem) <br>
newmail.Subject= 'Testing Mail' <br>
newmail.To='xyz@example.com' <br>
newmail.CC='xyz@example.com' <br>
newmail.Body= 'Hello, this is a test email.' <br>

##### attach='C:\\Users\\admin\\Desktop\\Python\\Sample.xlsx'
##### newmail.Attachments.Add(attach)

##### To display the mail before sending it
##### newmail.Display() 

newmail.Send()

**Reading Emails** <br>
import win32com.client

outlook = win32com.client.Dispatch("Outlook.Application")
inbox = outlook.GetNamespace("MAPI").GetDefaultFolder(6)
messages = inbox.Items

for message in messages:
    print("Subject:", message.Subject)
    print("Sender:", message.SenderName)
    print("Received Time:", message.ReceivedTime)
    print("Body:", message.Body)
    print("Attachments:", len(message.Attachments))
    print("----------------------")

**Managing Attachments** <br>
import win32com.client

outlook = win32com.client.Dispatch("Outlook.Application")
inbox = outlook.GetNamespace("MAPI").GetDefaultFolder(6)
messages = inbox.Items

for message in messages:
    for attachment in message.Attachments:
        attachment.SaveAsFile("C:/Attachments/" + attachment.FileName)
        print("Attachment saved:", attachment.FileName)

**Scheduling Emails** <br>
import win32com.client
import datetime

outlook = win32com.client.Dispatch("Outlook.Application")
mail = outlook.CreateItem(0)
mail.To = "recipient@example.com"
mail.Subject = "Scheduled Email"
mail.Body = "This email was scheduled to be sent at a specific time."
mail.Send()

schedule_time = datetime.datetime.now() + datetime.timedelta(minutes=5)
mail.DeferredDeliveryTime = schedule_time
mail.Send()

# ChatGPT

In [ ]:
import os
import requests
from urllib.parse import urlencode
from dotenv import load_dotenv
import msal

load_dotenv()

TENANT_ID = os.environ["TENANT_ID"]
CLIENT_ID = os.environ["CLIENT_ID"]
CLIENT_SECRET = os.environ["CLIENT_SECRET"]
SITE_ID = os.environ["SITE_ID"]
LIST_ID = os.environ["LIST_ID"]
SENDER_UPN = os.environ["SENDER_UPN"]

AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = ["https://graph.microsoft.com/.default"]
GRAPH = "https://graph.microsoft.com/v1.0"

# Adjust these to your actual SharePoint internal column names.
# Display names like "Status" and "NJ" may have different internal names (e.g., Status, NJ).
STATUS_FIELD = "Status"   # internal name for "status" column
EMAIL_FIELD = "NJ"        # internal name for "NJ" (email) column
NOTIFIED_FIELD = "Notified"  # boolean yes/no column you added

def get_token():
    app = msal.ConfidentialClientApplication(
        client_id=CLIENT_ID,
        client_credential=CLIENT_SECRET,
        authority=AUTHORITY
    )
    result = app.acquire_token_silent(SCOPE, account=None)
    if not result:
        result = app.acquire_token_for_client(scopes=SCOPE)
    if "access_token" not in result:
        raise RuntimeError(f"Token error: {result}")
    return result["access_token"]

def graph_get(url, token):
    r = requests.get(url, headers={"Authorization": f"Bearer {token}"})
    r.raise_for_status()
    return r.json()

def graph_patch(url, token, json):
    r = requests.patch(url, json=json, headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    })
    r.raise_for_status()
    return r.json() if r.text else {}

def graph_post(url, token, json):
    r = requests.post(url, json=json, headers={
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    })
    # sendMail returns 202 with empty body
    if r.status_code not in (200, 201, 202, 204):
        raise RuntimeError(f"POST {url} failed: {r.status_code} {r.text}")
    return r.json() if r.text else {}

def list_items_to_notify(token):
    """
    Pull items where:
      - Status == 'YES'
      - Notified != true (either false or missing)
    We select fields to keep payload small.
    """
    base = f"{GRAPH}/sites/{SITE_ID}/lists/{LIST_ID}/items"
    # Graph filter across list fields uses fields/<InternalName>
    params = {
        # filter for Status == 'YES' and (Notified ne true or null)
        "$filter": f"(fields/{STATUS_FIELD} eq 'YES') and (fields/{NOTIFIED_FIELD} ne true)",
        "$select": "id,fields",
        # Expand fields to read actual column values
        "$expand": "fields($select=Id,{0},{1},{2})".format(STATUS_FIELD, EMAIL_FIELD, NOTIFIED_FIELD)
    }
    url = f"{base}?{urlencode(params)}"

    items = []
    while url:
        data = graph_get(url, token)
        items.extend(data.get("value", []))
        url = data.get("@odata.nextLink")

    return items

def send_mail(token, to_email, subject, html_body):
    url = f"{GRAPH}/users/{SENDER_UPN}/sendMail"
    payload = {
        "message": {
            "subject": subject,
            "body": {"contentType": "HTML", "content": html_body},
            "toRecipients": [{"emailAddress": {"address": to_email}}],
        },
        "saveToSentItems": True
    }
    graph_post(url, token, payload)

def mark_notified(token, item_id):
    """
    PATCH the item's fields to set Notified = true
    """
    url = f"{GRAPH}/sites/{SITE_ID}/lists/{LIST_ID}/items/{item_id}/fields"
    graph_patch(url, token, {NOTIFIED_FIELD: True})

def main():
    token = get_token()
    items = list_items_to_notify(token)

    for it in items:
        fields = it.get("fields", {})
        to_addr = fields.get(EMAIL_FIELD)
        status_val = fields.get(STATUS_FIELD)

        if not to_addr:
            # Skip rows without NJ email
            continue

        subject = "Notification: Status is YES"
        body = f"""
            <p>Hello,</p>
            <p>This is an automated notification. The SharePoint item with ID <b>{fields.get('Id')}</b>
            has <b>Status = {status_val}</b>.</p>
            <p>Thank you.</p>
        """

        # 1) Send the email
        send_mail(token, to_addr, subject, body)

        # 2) Mark as notified to avoid duplicates
        mark_notified(token, it["id"])

    print(f"Processed {len(items)} item(s).")

if __name__ == "__main__":
    main()


# Gemini

In [ ]:
from office365.sharepoint.client import ClientContext
from office365.runtime.auth.user_credential import UserCredential
from office365.runtime.auth.client_credential import ClientCredential
from email.message import EmailMessage
import smtplib

# --- SharePoint Configuration ---
sharepoint_url = "https://yourtenant.sharepoint.com/sites/yoursite"
list_name = "Your List Name"
username = "your_username@yourtenant.com"
password = "your_password"

# --- Email Configuration ---
sender_email = "your_email@yourtenant.com"
sender_password = "your_email_password" # Or use an application-specific password
smtp_server = "smtp.office365.com"
smtp_port = 587

def send_email(receiver_email, subject, body):
    """Sends an email using an SMTP server."""
    msg = EmailMessage()
    msg.set_content(body)
    msg["Subject"] = subject
    msg["From"] = sender_email
    msg["To"] = receiver_email

    try:
        with smtplib.SMTP(smtp_server, smtp_port) as server:
            server.starttls()  # Secure the connection
            server.login(sender_email, sender_password)
            server.send_message(msg)
            print(f"Email sent successfully to {receiver_email}")
    except Exception as e:
        print(f"Failed to send email to {receiver_email}. Error: {e}")

try:
    # Authenticate and connect to SharePoint
    user_credentials = UserCredential(username, password)
    ctx = ClientContext(sharepoint_url).with_credentials(user_credentials)

    # Get the SharePoint list
    sharepoint_list = ctx.web.lists.get_by_title(list_name)

    # Filter items where 'status' is 'YES'
    # Note: Column names are case-sensitive and might require internal names
    # You can check the internal name in List Settings -> Column -> URL
    items = sharepoint_list.items.filter("status eq 'YES'").get().execute_query()

    if not items:
        print("No items with 'status' set to 'YES' found.")
    else:
        for item in items:
            # Get the email address from the 'NJ' column
            # Replace 'NJ' with the actual internal name of your column if different
            email_address = item.properties.get("NJ")
            if email_address:
                subject = "Your Subject Here"
                body = "Your email body content here."
                send_email(email_address, subject, body)
            else:
                print(f"Skipping item with ID {item.id}: 'NJ' column is empty.")

except Exception as e:
    print(f"An error occurred: {e}")

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 20.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# DeepSeek

In [11]:
import requests
from requests_ntlm import HttpNtlmAuth
from dotenv import load_dotenv
import os
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Load environment variables
load_dotenv()

# 1. CONFIGURATION (Load from .env file)
SHAREPOINT_SITE_URL = os.getenv('SHAREPOINT_SITE_URL')
SHAREPOINT_LIST_NAME = "YourListName"  # Change this to your list's name
SHAREPOINT_USERNAME = os.getenv('SHAREPOINT_USERNAME')
SHAREPOINT_PASSWORD = os.getenv('SHAREPOINT_PASSWORD')

# Email Configuration (Update with your SMTP details)
SMTP_SERVER = "smtp.office365.com"  # For Outlook/Office 365. Use smtp.gmail.com for Gmail.
SMTP_PORT = 587
EMAIL_USERNAME = os.getenv('SHAREPOINT_USERNAME')  # Often the same username
EMAIL_PASSWORD = os.getenv('SHAREPOINT_PASSWORD')  # Often the same password/app password

# 2. AUTHENTICATE AND FETCH DATA FROM SHAREPOINT
def get_sharepoint_list_items():
    """
    Fetches items from the specified SharePoint list.
    """
    # Construct the API endpoint URL for the list items
    list_api_endpoint = f"{SHAREPOINT_SITE_URL}/_api/web/lists/getbytitle('{SHAREPOINT_LIST_NAME}')/items"
    
    # Set up authentication and headers
    auth = HttpNtlmAuth(SHAREPOINT_USERNAME, SHAREPOINT_PASSWORD)
    headers = {
        'Accept': 'application/json;odata=verbose',
        'Content-Type': 'application/json;odata=verbose',
    }
    
    try:
        # Make the GET request
        response = requests.get(list_api_endpoint, auth=auth, headers=headers)
        response.raise_for_status()  # Raises an exception for bad status codes (4xx or 5xx)
        
        # Parse the JSON response
        data = response.json()
        # Navigate the nested structure to get the actual items
        items = data['d']['results']
        print(f"Successfully retrieved {len(items)} items.")
        return items
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from SharePoint: {e}")
        return []

# 3. SEND EMAIL FUNCTION
def send_email(to_email, subject, body):
    """
    Sends an email using SMTP.
    """
    msg = MIMEMultipart()
    msg['From'] = EMAIL_USERNAME
    msg['To'] = to_email
    msg['Subject'] = subject
    
    # Attach the body text
    msg.attach(MIMEText(body, 'plain'))
    
    try:
        # Start SMTP session
        server = smtplib.SMTP(SMTP_SERVER, SMTP_PORT)
        server.starttls()  # Upgrade connection to secure TLS
        server.login(EMAIL_USERNAME, EMAIL_PASSWORD)
        text = msg.as_string()
        server.sendmail(EMAIL_USERNAME, to_email, text)
        server.quit()
        print(f"Email successfully sent to {to_email}")
        return True
    except Exception as e:
        print(f"Failed to send email to {to_email}: {e}")
        return False

# 4. MAIN LOGIC
def main():
    # Get all items from the SharePoint list
    list_items = get_sharepoint_list_items()
    
    # Counter for emails sent
    emails_sent = 0
    
    # Loop through each item in the list
    for item in list_items:
        # Get the values from the columns. Adjust field names as needed.
        # Internal field names are often different from the display name.
        # Common internal names: "Status", "NJ", or "NJ0" (if it's a person field, it's more complex).
        status = item.get('Status')  # Check the exact internal name in your API response
        nj_email = item.get('NJ')    # Check the exact internal name
        
        # Check if this row meets our condition
        if status and status.upper() == "YES" and nj_email:
            print(f"Processing item ID {item['Id']}: Status is YES. Sending email to {nj_email}...")
            
            # Define email content
            email_subject = "Action Required: Your Status is YES"
            email_body = f"Hello,\n\nThis is an automated notification because your status was set to 'YES' in the SharePoint list.\n\nBest regards,\nYour Automation Script"
            
            # Send the email
            success = send_email(nj_email, email_subject, email_body)
            if success:
                emails_sent += 1
        # else:
        #     print(f"Skipping item ID {item['Id']}: Status is not YES or NJ email is missing.")
    
    print(f"Automation run complete. {emails_sent} emails were sent.")

if __name__ == "__main__":
    main()

Error fetching data from SharePoint: HTTPSConnectionPool(host='yourcompany.sharepoint.com', port=443): Max retries exceeded with url: /sites/yoursitename/_api/web/lists/getbytitle('YourListName')/items (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001DFC8573B60>: Failed to resolve 'yourcompany.sharepoint.com' ([Errno 11001] getaddrinfo failed)"))
Automation run complete. 0 emails were sent.
